# Kaggle Smoke Test for PoRT LLM Unlearning Experiment

This notebook clones the public GitHub repository, checks the Kaggle-ready project paths, loads TOFU/WMDP samples, and runs a full-shape smoke flow through the project dataset/evaluator/inference classes. It intentionally uses fake model/tokenizer objects so the run stays small and catches syntax/interface/logic errors without downloading a large LLM checkpoint.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import re
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()

def has_project_layout(path):
    path = Path(path)
    return (path / 'llm-unlearn-eco' / 'eco').exists() and (path / 'dataset').exists()

def refresh_existing_clone(path):
    path = Path(path)
    if (path / '.git').exists():
        print(f'Refreshing existing clone: {path}')
        subprocess.check_call(['git', '-C', str(path), 'pull', '--ff-only'])

def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            refresh_existing_clone(target)
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        print(f'Using local checkout: {local_root}')
        return local_root

    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        print(f'Using existing cloned repository: {target}')
        return target.resolve()
    print(f'Cloning {REPO_URL} into {target}')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()

PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
TOFU_DIR = PROJECT_ROOT / 'dataset' / 'TOFU' / 'original'
WMDP_DIR = PROJECT_ROOT / 'dataset' / 'WMDP' / 'original'
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)

if str(ECO_ROOT) not in sys.path:
    sys.path.insert(0, str(ECO_ROOT))

logic_warnings = []
print(f'Project root: {PROJECT_ROOT}')
print(f'ECO root:     {ECO_ROOT}')
print(f'TOFU data:    {TOFU_DIR}')
print(f'WMDP data:    {WMDP_DIR}')


In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'rouge_score': 'rouge-score>=0.1.2',
    'tabulate': 'tabulate',
    'tqdm': 'tqdm',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing smoke-test packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required smoke-test packages are already available.')


## Source, Path, and Package Checks


In [ ]:
source_roots = [
    ECO_ROOT / 'eco',
    ECO_ROOT / 'scripts',
    PROJECT_ROOT / 'PoRT_pipeline',
    PROJECT_ROOT / 'post_classifier',
]
python_files = sorted({path for root in source_roots if root.exists() for path in root.rglob('*.py')})

syntax_failures = []
for path in python_files:
    try:
        compile(path.read_text(encoding='utf-8'), str(path), 'exec')
    except Exception as exc:
        syntax_failures.append({'file': str(path.relative_to(PROJECT_ROOT)), 'error': repr(exc)})

print(f'Compiled {len(python_files)} Python files.')
if syntax_failures:
    print(json.dumps(syntax_failures, indent=2))
    raise SyntaxError('Smoke test found Python syntax/compile failures.')
print('Syntax compile check passed.')

placeholder_hits = {}
placeholder_pattern = re.compile(r'<[A-Z0-9_/-]+>')
for path in python_files:
    hits = sorted(set(placeholder_pattern.findall(path.read_text(encoding='utf-8'))))
    if hits:
        placeholder_hits[str(path.relative_to(PROJECT_ROOT))] = hits

if placeholder_hits:
    logic_warnings.append('Placeholder paths still exist in Python files.')
    print(json.dumps(placeholder_hits, indent=2))
else:
    print('No placeholder paths found in Python files.')


In [ ]:
modules_to_try = ['eco.paths', 'eco.dataset', 'eco.evaluator', 'eco.inference', 'eco.utils']
package_import_results = {}
for module_name in modules_to_try:
    try:
        importlib.import_module(module_name)
        package_import_results[module_name] = 'OK'
    except Exception as exc:
        package_import_results[module_name] = f'{type(exc).__name__}: {exc}'

print(json.dumps(package_import_results, indent=2))
failed_package_imports = {k: v for k, v in package_import_results.items() if v != 'OK'}
if failed_package_imports:
    raise ImportError(f'Package imports failed: {failed_package_imports}')

from datasets import DatasetDict
from eco.dataset import TOFU, TOFUPerturbed, WMDPBio, WMDPChem, WMDPCyber
from eco.evaluator import ChoiceByTopLogit, ROUGERecall
from eco.inference import EvaluationEngine, GenerationEngine
from eco.paths import MODEL_CONFIG_DIR, RESULTS_DIR, TASK_CONFIG_DIR, TOFU_DATASET_DIR, WMDP_DATASET_DIR

print('Imported project components successfully.')


## TOFU Dataset Smoke Tests


In [ ]:
required_tofu_files = [
    'forget01.json', 'forget05.json', 'forget10.json',
    'retain90.json', 'retain95.json', 'retain99.json',
    'real_authors.json', 'world_facts.json',
    'forget01_perturbed.json', 'retain_perturbed.json',
]
missing_tofu_files = [name for name in required_tofu_files if not (TOFU_DIR / name).exists()]
assert not missing_tofu_files, f'Missing TOFU files: {missing_tofu_files}'

tofu_file_counts = {}
for name in required_tofu_files:
    data = json.loads((TOFU_DIR / name).read_text(encoding='utf-8'))
    assert isinstance(data, list) and data, f'{name} must be a non-empty JSON list.'
    assert {'question', 'answer'}.issubset(data[0]), f'{name} first row must contain question and answer.'
    tofu_file_counts[name] = len(data)

formatting_tokens = {
    'prompt_prefix': 'Question: ',
    'prompt_suffix': '\nAnswer:',
    'answer_prefix': ' ',
    'answer_suffix': '',
}

tofu = TOFU(formatting_tokens=formatting_tokens, eos_token='')
tofu.download()
tofu_counts = {name: len(tofu.dataset[name]) for name in tofu.subsets}
assert all(count > 0 for count in tofu_counts.values()), tofu_counts

eval_dataset = tofu.load_dataset_for_eval('forget01')
assert {'prompt', 'answer', 'prompt_formatted'}.issubset(set(eval_dataset.column_names)), eval_dataset.column_names
tofu_eval_samples = [eval_dataset[i] for i in range(min(3, len(eval_dataset)))]
assert all(row['prompt_formatted'].startswith('Question: ') for row in tofu_eval_samples)

batched_eval = tofu.load_dataset_for_eval('forget01', load_in_batch=True, batch_size=2)
assert batched_eval and len(batched_eval[0]['prompt']) <= 2

classification_dataset = tofu.load_dataset_for_classification('forget01', use_val=True)
classification_counts = {split: len(classification_dataset[split]) for split in classification_dataset.keys()}
train_labels = set(classification_dataset['train']['label'])
assert train_labels.issubset({0, 1}) and train_labels, train_labels

print('TOFU file counts:', tofu_file_counts)
print('TOFU subset counts:', tofu_counts)
print('TOFU classification counts:', classification_counts)
print(json.dumps(tofu_eval_samples, indent=2)[:2000])


In [ ]:
tofu_perturbed = TOFUPerturbed(formatting_tokens, '')
perturbed_eval = tofu_perturbed.load_dataset_for_eval('forget01_perturbed')
assert {'prompt', 'answer', 'perturbed_answer', 'choices', 'prompt_formatted'}.issubset(set(perturbed_eval.column_names)), perturbed_eval.column_names
perturbed_sample = perturbed_eval[0]
assert isinstance(perturbed_sample['choices'], list) and len(perturbed_sample['choices']) >= 2
assert perturbed_sample['prompt_formatted'].startswith('Question: '), perturbed_sample['prompt_formatted']

print('TOFUPerturbed first sample:')
print(json.dumps({
    'prompt_formatted': perturbed_sample['prompt_formatted'],
    'answer': perturbed_sample['answer'],
    'num_choices': len(perturbed_sample['choices']),
}, indent=2)[:2000])


## WMDP Dataset Smoke Tests


In [ ]:
wmdp_modules = [WMDPBio(sample_size=3), WMDPChem(sample_size=3), WMDPCyber(sample_size=3)]
wmdp_samples = {}
for data_module in wmdp_modules:
    dataset = data_module.load_dataset_for_eval('test')
    assert len(dataset) == 3, (data_module.name, len(dataset))
    assert {'prompt', 'choices', 'correct_answer'}.issubset(set(dataset.column_names)), dataset.column_names
    row = dataset[0]
    assert row['choices'] == ['A', 'B', 'C', 'D'], row['choices']
    assert isinstance(row['correct_answer'], int) and 0 <= row['correct_answer'] <= 3, row['correct_answer']
    assert 'A.' in row['prompt'] and 'Answer:' in row['prompt'], row['prompt'][:500]
    batched = data_module.load_dataset_for_eval('test', load_in_batch=True, batch_size=2)
    assert batched and len(batched[0]['prompt']) <= 2
    wmdp_samples[data_module.name] = {
        'rows': len(dataset),
        'first_correct_answer': row['correct_answer'],
        'first_prompt': row['prompt'][:800],
    }

print('WMDP smoke samples:')
print(json.dumps(wmdp_samples, indent=2)[:3000])


## Full-Shape Sample Smoke Run

This section uses the same `EvaluationEngine` and `GenerationEngine` classes as full runs, but with two or three examples and fake model/tokenizer objects. It verifies data-module, batching, evaluator, inference, summary, and output-writing interfaces without loading a real LLM.


In [ ]:
import types
import torch

class TensorBatch(dict):
    def __getattr__(self, name):
        return self[name]

    def to(self, device):
        return self

class FakeTokenizer:
    padding_side = 'right'
    pad_token_id = 0
    eos_token_id = 2
    eos_token = ''
    choice_to_id = {'A': 10, 'B': 11, 'C': 12, 'D': 13}

    def __init__(self):
        self.last_texts = []
        self.next_decode_texts = None

    def __call__(self, texts, padding=None, return_tensors=None, add_special_tokens=True, truncation=False, max_length=None):
        if isinstance(texts, str):
            texts = [texts]
        texts = list(texts)
        self.last_texts = texts
        if texts and all(text in self.choice_to_id for text in texts):
            input_ids = [[self.choice_to_id[text]] for text in texts]
            attention_mask = [[1] for _ in texts]
        else:
            lengths = [max(1, min(len(text.split()), max_length or 256)) for text in texts]
            max_len = max(lengths) if lengths else 1
            input_ids = [[1] * length + [0] * (max_len - length) for length in lengths]
            attention_mask = [[1] * length + [0] * (max_len - length) for length in lengths]
        return TensorBatch({
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        })

    def encode(self, text):
        return self(text)['input_ids'][0].tolist()

    def batch_decode(self, ids, skip_special_tokens=True):
        if self.next_decode_texts is not None:
            texts = self.next_decode_texts
            self.next_decode_texts = None
            return texts
        return list(self.last_texts)

class EmptyTorchModule(torch.nn.Module):
    def __init__(self):
        super().__init__()

class FakeOutput:
    def __init__(self, logits):
        self.logits = logits

class FakeModel:
    model_name = 'fake-smoke-model'
    device = 'cpu'
    generation_config = types.SimpleNamespace(max_new_tokens=8, do_sample=False)

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.model = EmptyTorchModule()
        self.model_config = {'formatting_tokens': formatting_tokens, 'attack_module': ''}

    def __call__(self, input_ids, attention_mask, prompts=None, answers=None, **kwargs):
        batch_size, seq_len = input_ids.shape
        logits = torch.zeros(batch_size, seq_len, 64)
        for i in range(batch_size):
            prompt_end = int(attention_mask[i].sum().item()) - 1 if self.tokenizer.padding_side == 'right' else -1
            logits[i, prompt_end, 10 + (i % 4)] = 10.0
        return FakeOutput(logits)

    def generate(self, input_ids, attention_mask=None, prompts=None, **kwargs):
        prompts = prompts or self.tokenizer.last_texts
        self.tokenizer.next_decode_texts = [prompt + ' smoke generated answer' for prompt in prompts]
        return torch.ones((len(prompts), 4), dtype=torch.long)

fake_tokenizer = FakeTokenizer()
fake_model = FakeModel(fake_tokenizer)
print('Fake model/tokenizer ready.')


In [ ]:
wmdp_full_shape = {}
for data_module in [WMDPBio(sample_size=2), WMDPChem(sample_size=2), WMDPCyber(sample_size=2)]:
    engine = EvaluationEngine(
        model=fake_model,
        tokenizer=fake_tokenizer,
        data_module=data_module,
        subset_names=['test'],
        evaluator=ChoiceByTopLogit(),
        batch_size=2,
    )
    engine.inference()
    summary_stats, outputs = engine.summary()
    wmdp_full_shape[data_module.name] = {
        'summary': summary_stats,
        'num_output_groups': len(outputs),
    }

print('WMDP full-shape sample summary:')
print(json.dumps(wmdp_full_shape, indent=2))


In [ ]:
tofu_sample = TOFU(formatting_tokens=formatting_tokens, eos_token='')
tofu_sample.download()
tofu_sample.dataset = DatasetDict({
    key: value.select(range(min(2, len(value))))
    for key, value in tofu_sample.dataset.items()
})

generation_engine = GenerationEngine(
    model=fake_model,
    tokenizer=fake_tokenizer,
    data_module=tofu_sample,
    subset_names=['forget01'],
    evaluator=ROUGERecall(mode='rougeL'),
    batch_size=2,
)
generation_engine.inference()
tofu_generation_summary, tofu_generation_outputs = generation_engine.summary()
assert generation_engine.text_generations['tofu_forget01']['generated'], generation_engine.text_generations

print('TOFU GenerationEngine sample summary:')
print(json.dumps(tofu_generation_summary, indent=2))
print(json.dumps(generation_engine.text_generations['tofu_forget01'], indent=2)[:2000])


## Smoke Summary Artifact


In [ ]:
output_dir = (Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT) / 'smoke_outputs'
output_dir.mkdir(parents=True, exist_ok=True)

smoke_summary = {
    'project_root': str(PROJECT_ROOT),
    'syntax_files_checked': len(python_files),
    'syntax_failures': syntax_failures,
    'placeholder_hits': placeholder_hits,
    'package_import_results': package_import_results,
    'logic_warnings': logic_warnings,
    'tofu_file_counts': tofu_file_counts,
    'tofu_counts': tofu_counts,
    'tofu_classification_counts': classification_counts,
    'tofu_eval_sample_count': len(tofu_eval_samples),
    'wmdp_samples': wmdp_samples,
    'wmdp_full_shape': wmdp_full_shape,
    'tofu_generation_summary': tofu_generation_summary,
}

summary_path = output_dir / 'smoke_summary.json'
summary_path.write_text(json.dumps(smoke_summary, indent=2, default=str), encoding='utf-8')

sample_path = output_dir / 'tofu_eval_samples.json'
sample_path.write_text(json.dumps(tofu_eval_samples, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'Wrote summary to: {summary_path}')
print(f'Wrote TOFU samples to: {sample_path}')
print(json.dumps(smoke_summary, indent=2, default=str)[:4000])
print('SMOKE TEST COMPLETED')
